# Mixture of Experts (MoE)

## Overview

**Mixture of Experts (MoE)** is a powerful technique for scaling neural networks efficiently. Instead of activating all parameters for every input, MoE models route each input to a subset of specialized "expert" networks. This enables:

- **Sparse computation**: Only a fraction of parameters are active per token
- **Scalability**: Train models with trillions of parameters without proportional compute costs
- **Specialization**: Different experts learn different patterns (e.g., one for code, one for math)

**Key Applications**:
- GPT-4 (rumored to use MoE with 8×220B experts)
- Switch Transformers (1.6T parameters, outperforms dense models)
- Mixture-of-Depths (sparse depth instead of width)

**Learning Objectives**:
1. Understand MoE architecture and routing mechanisms
2. Implement gating networks with top-k selection
3. Design load balancing loss to prevent expert collapse
4. Build Switch Transformers with expert FFN layers
5. Address training challenges (load imbalance, routing instability)

---

## Table of Contents
1. [MoE Fundamentals](#1.-MoE-Fundamentals)
2. [Routing Algorithms](#2.-Routing-Algorithms)
3. [Switch Transformers](#3.-Switch-Transformers)
4. [Load Balancing](#4.-Load-Balancing)
5. [Training Challenges](#5.-Training-Challenges)
6. [Advanced MoE Variants](#6.-Advanced-MoE-Variants)
7. [Exercises](#7.-Exercises)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Tuple, Optional
from dataclasses import dataclass

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. MoE Fundamentals

### 1.1 Core Concept

**Standard Dense Layer**:
```
y = σ(Wx + b)
All weights W are used for every input x
Compute: O(d_in × d_out)
```

**MoE Layer**:
```
1. Routing: g = Gating(x)  → scores for each expert
2. Selection: top_k = TopK(g, k)  → choose k best experts
3. Compute: y = Σᵢ gᵢ · Expertᵢ(x)  → weighted sum of expert outputs

Only k experts are activated (typically k=1 or k=2)
Compute: O(k × d_in × d_out / num_experts)
```

**Benefits**:
- **Sparse activation**: Only 1/num_experts of parameters active
- **Conditional computation**: Different inputs use different experts
- **Model capacity**: Can have many experts without proportional compute

**Example**: Model with 8 experts, k=1:
- Total parameters: 8× a single expert
- Compute per token: 1× a single expert
- Effective scaling: 8× model capacity at 1× compute!

In [ ]:
# Visualize MoE concept
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Dense layer
ax = axes[0]
ax.set_title('Dense Layer', fontsize=14, fontweight='bold')
ax.text(0.5, 0.8, 'Input', ha='center', fontsize=12, bbox=dict(boxstyle='round', facecolor='lightblue'))
ax.text(0.5, 0.5, 'All Parameters\nActivated', ha='center', fontsize=11, bbox=dict(boxstyle='round', facecolor='lightcoral'))
ax.text(0.5, 0.2, 'Output', ha='center', fontsize=12, bbox=dict(boxstyle='round', facecolor='lightgreen'))
ax.arrow(0.5, 0.75, 0, -0.15, head_width=0.05, head_length=0.05, fc='black', ec='black')
ax.arrow(0.5, 0.4, 0, -0.15, head_width=0.05, head_length=0.05, fc='black', ec='black')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')
ax.text(0.5, 0.05, 'Compute: O(d_in × d_out)', ha='center', fontsize=10, style='italic')

# MoE layer
ax = axes[1]
ax.set_title('Mixture of Experts (k=2)', fontsize=14, fontweight='bold')
ax.text(0.5, 0.9, 'Input', ha='center', fontsize=12, bbox=dict(boxstyle='round', facecolor='lightblue'))
ax.text(0.5, 0.75, 'Gating Network', ha='center', fontsize=10, bbox=dict(boxstyle='round', facecolor='lightyellow'))

experts_x = [0.15, 0.35, 0.55, 0.75]
expert_labels = ['Expert 1', 'Expert 2\n(selected)', 'Expert 3', 'Expert 4\n(selected)']
expert_colors = ['lightgray', 'lightcoral', 'lightgray', 'lightcoral']
for i, (x, label, color) in enumerate(zip(experts_x, expert_labels, expert_colors)):
    ax.text(x, 0.5, label, ha='center', fontsize=9, bbox=dict(boxstyle='round', facecolor=color))
    if i in [1, 3]:  # Selected experts
        ax.arrow(0.5, 0.7, x-0.5, -0.15, head_width=0.02, head_length=0.02, fc='red', ec='red', linewidth=2)
    else:
        ax.arrow(0.5, 0.7, x-0.5, -0.15, head_width=0.02, head_length=0.02, fc='gray', ec='gray', linewidth=0.5, linestyle='--')

ax.text(0.5, 0.2, 'Weighted Sum', ha='center', fontsize=11, bbox=dict(boxstyle='round', facecolor='lightgreen'))
for x in [0.35, 0.75]:
    ax.arrow(x, 0.45, 0.5-x, -0.2, head_width=0.02, head_length=0.02, fc='red', ec='red', linewidth=1.5)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')
ax.text(0.5, 0.05, 'Compute: O(k × d_in × d_out / num_experts)', ha='center', fontsize=10, style='italic')

plt.tight_layout()
plt.savefig('moe_architecture.png', dpi=150, bbox_inches='tight')
plt.show()

print("MoE enables sparse computation by routing inputs to specialized experts.")

### 1.2 Basic MoE Implementation

In [ ]:
class Expert(nn.Module):
    """Single expert network (typically a FFN)."""
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class TopKGating(nn.Module):
    """Gating network with top-k expert selection."""
    def __init__(self, d_model: int, num_experts: int, k: int = 2, noisy_gating: bool = True):
        super().__init__()
        self.num_experts = num_experts
        self.k = k
        self.noisy_gating = noisy_gating
        
        # Gating network: linear layer to produce expert scores
        self.w_gate = nn.Linear(d_model, num_experts, bias=False)
        
        # Noise for exploration (helps with load balancing)
        if noisy_gating:
            self.w_noise = nn.Linear(d_model, num_experts, bias=False)
    
    def forward(self, x: torch.Tensor, train: bool = True) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Args:
            x: Input tensor (batch_size, seq_len, d_model)
            train: Whether in training mode (for noise)
        
        Returns:
            gates: Gating weights for top-k experts (batch_size, seq_len, k)
            indices: Indices of top-k experts (batch_size, seq_len, k)
            load: Load on each expert for load balancing (num_experts,)
        """
        # Compute gating scores
        clean_logits = self.w_gate(x)  # (batch, seq_len, num_experts)
        
        # Add noise during training for exploration
        if self.noisy_gating and train:
            noise_stddev = 1.0 / self.num_experts
            noise_logits = self.w_noise(x)
            noise = torch.randn_like(clean_logits) * F.softplus(noise_logits) * noise_stddev
            logits = clean_logits + noise
        else:
            logits = clean_logits
        
        # Top-k selection
        top_k_logits, top_k_indices = torch.topk(logits, k=self.k, dim=-1)
        
        # Softmax over top-k experts only
        top_k_gates = F.softmax(top_k_logits, dim=-1)
        
        # Compute load on each expert (for load balancing loss)
        # Count how many tokens are assigned to each expert
        gates_full = torch.zeros_like(logits).scatter_(-1, top_k_indices, top_k_gates)
        load = gates_full.sum(dim=[0, 1])  # Sum over batch and sequence
        
        return top_k_gates, top_k_indices, load


class SimpleMoE(nn.Module):
    """Simple Mixture of Experts layer."""
    def __init__(self, d_model: int, d_ff: int, num_experts: int, k: int = 2):
        super().__init__()
        self.num_experts = num_experts
        self.k = k
        
        # Create experts
        self.experts = nn.ModuleList([Expert(d_model, d_ff) for _ in range(num_experts)])
        
        # Gating network
        self.gate = TopKGating(d_model, num_experts, k)
    
    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Args:
            x: Input tensor (batch_size, seq_len, d_model)
        
        Returns:
            output: MoE output (batch_size, seq_len, d_model)
            gates: Gating weights
            load: Expert load for loss
        """
        batch_size, seq_len, d_model = x.shape
        
        # Get top-k experts and their gates
        gates, indices, load = self.gate(x, train=self.training)
        
        # Flatten for expert computation
        x_flat = x.view(-1, d_model)  # (batch * seq_len, d_model)
        gates_flat = gates.view(-1, self.k)  # (batch * seq_len, k)
        indices_flat = indices.view(-1, self.k)  # (batch * seq_len, k)
        
        # Initialize output
        output = torch.zeros_like(x_flat)
        
        # Compute expert outputs for selected experts
        for i in range(self.k):
            expert_indices = indices_flat[:, i]  # Which expert for each token
            expert_weights = gates_flat[:, i:i+1]  # Weight for this expert
            
            # Process each expert separately (can be batched for efficiency)
            for expert_id in range(self.num_experts):
                # Find tokens assigned to this expert
                mask = (expert_indices == expert_id)
                if mask.any():
                    # Get tokens for this expert
                    expert_input = x_flat[mask]
                    
                    # Compute expert output
                    expert_output = self.experts[expert_id](expert_input)
                    
                    # Weight by gate and add to output
                    output[mask] += expert_weights[mask] * expert_output
        
        # Reshape back
        output = output.view(batch_size, seq_len, d_model)
        
        return output, gates, load


# Test SimpleMoE
print("=== Simple MoE Test ===")
d_model, d_ff, num_experts, k = 512, 2048, 8, 2
batch_size, seq_len = 4, 16

moe = SimpleMoE(d_model, d_ff, num_experts, k).to(device)
x = torch.randn(batch_size, seq_len, d_model).to(device)

output, gates, load = moe(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Gates shape: {gates.shape}")
print(f"Expert load: {load.cpu().numpy()}")
print(f"\nLoad balance: min={load.min().item():.1f}, max={load.max().item():.1f}, mean={load.mean().item():.1f}")

# Count parameters
total_params = sum(p.numel() for p in moe.parameters())
expert_params = sum(p.numel() for p in moe.experts[0].parameters())
print(f"\nTotal parameters: {total_params:,}")
print(f"Per expert: {expert_params:,}")
print(f"Active per token (k={k}): {k * expert_params:,} ({100 * k / num_experts:.1f}% of total experts)")

## 2. Routing Algorithms

### 2.1 Gating Strategies

Several algorithms for routing tokens to experts:

**1. Top-K Gating** (most common):
```
logits = W_g · x
top_k_indices = topK(logits, k)
gates = softmax(logits[top_k_indices])
output = Σ gates[i] · Expert[top_k_indices[i]](x)
```

**2. Switch Routing** (k=1, simpler):
```
logits = W_g · x
expert_id = argmax(logits)
output = Expert[expert_id](x)
```

**3. Soft Routing** (all experts, weighted):
```
logits = W_g · x
gates = softmax(logits)
output = Σ gates[i] · Expert[i](x)
```

**4. Expert Choice** (experts choose tokens):
```
# Instead of tokens choosing experts, experts choose tokens
For each expert:
    scores = Expert.selector · all_tokens
    top_k_tokens = topK(scores, capacity)
    process those tokens
```

**Comparison**:
- **Top-K**: Balances specialization (k small) and robustness (k>1)
- **Switch**: Simplest, most efficient, but less robust (all load on 1 expert)
- **Soft**: No sparsity benefit, but differentiable and stable
- **Expert Choice**: Better load balancing, but more complex

In [ ]:
# Visualize different routing strategies
np.random.seed(42)
num_tokens, num_experts = 100, 8
logits = np.random.randn(num_tokens, num_experts)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Top-2 gating
ax = axes[0, 0]
top_k = 2
top_k_indices = np.argsort(logits, axis=1)[:, -top_k:]
assignment = np.zeros_like(logits)
for i in range(num_tokens):
    assignment[i, top_k_indices[i]] = 1
ax.imshow(assignment.T, aspect='auto', cmap='Reds', interpolation='none')
ax.set_xlabel('Token Index', fontsize=11)
ax.set_ylabel('Expert ID', fontsize=11)
ax.set_title('Top-2 Gating (k=2)', fontsize=12, fontweight='bold')
ax.set_yticks(range(num_experts))
expert_load = assignment.sum(axis=0)
ax.text(0.5, -0.15, f'Load: min={expert_load.min():.0f}, max={expert_load.max():.0f}, std={expert_load.std():.1f}', 
        ha='center', transform=ax.transAxes, fontsize=10)

# Switch routing (k=1)
ax = axes[0, 1]
top_1_indices = np.argmax(logits, axis=1)
assignment_switch = np.zeros_like(logits)
for i in range(num_tokens):
    assignment_switch[i, top_1_indices[i]] = 1
ax.imshow(assignment_switch.T, aspect='auto', cmap='Blues', interpolation='none')
ax.set_xlabel('Token Index', fontsize=11)
ax.set_ylabel('Expert ID', fontsize=11)
ax.set_title('Switch Routing (k=1)', fontsize=12, fontweight='bold')
ax.set_yticks(range(num_experts))
expert_load = assignment_switch.sum(axis=0)
ax.text(0.5, -0.15, f'Load: min={expert_load.min():.0f}, max={expert_load.max():.0f}, std={expert_load.std():.1f}', 
        ha='center', transform=ax.transAxes, fontsize=10)

# Soft routing (all experts)
ax = axes[1, 0]
from scipy.special import softmax
gates_soft = softmax(logits, axis=1)
ax.imshow(gates_soft.T, aspect='auto', cmap='Greens', interpolation='none', vmin=0, vmax=1)
ax.set_xlabel('Token Index', fontsize=11)
ax.set_ylabel('Expert ID', fontsize=11)
ax.set_title('Soft Routing (all experts weighted)', fontsize=12, fontweight='bold')
ax.set_yticks(range(num_experts))
expert_load = gates_soft.sum(axis=0)
ax.text(0.5, -0.15, f'Load: min={expert_load.min():.1f}, max={expert_load.max():.1f}, std={expert_load.std():.2f}', 
        ha='center', transform=ax.transAxes, fontsize=10)

# Expert load distribution comparison
ax = axes[1, 1]
load_topk = assignment.sum(axis=0)
load_switch = assignment_switch.sum(axis=0)
load_soft = gates_soft.sum(axis=0)

x = np.arange(num_experts)
width = 0.25
ax.bar(x - width, load_topk, width, label='Top-2', color='red', alpha=0.7)
ax.bar(x, load_switch, width, label='Switch', color='blue', alpha=0.7)
ax.bar(x + width, load_soft, width, label='Soft', color='green', alpha=0.7)
ax.set_xlabel('Expert ID', fontsize=11)
ax.set_ylabel('Total Load', fontsize=11)
ax.set_title('Expert Load Distribution', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('routing_strategies.png', dpi=150, bbox_inches='tight')
plt.show()

print("Different routing strategies show varying levels of sparsity and load balance.")
print(f"Top-2 std: {load_topk.std():.2f}, Switch std: {load_switch.std():.2f}, Soft std: {load_soft.std():.2f}")

### 2.2 Capacity Factor and Dropping

**Problem**: What if too many tokens are routed to one expert?

**Solution**: **Expert capacity** limits tokens per expert:

```
capacity = (total_tokens / num_experts) × capacity_factor

capacity_factor = 1.0: Each expert processes exactly its "fair share"
capacity_factor = 1.25: 25% buffer for imbalance
capacity_factor = 2.0: Double capacity (more robust but less efficient)
```

**Dropping mechanism**:
1. Each expert has a buffer of size `capacity`
2. Tokens arrive in order of gating scores (highest first)
3. Once an expert's buffer is full, remaining tokens are **dropped**
4. Dropped tokens either:
   - Pass through unchanged (identity)
   - Routed to next-best expert
   - Processed by auxiliary "overflow" expert

**Trade-offs**:
- **Low capacity_factor** (1.0-1.25): Efficient, but may drop many tokens
- **High capacity_factor** (1.5-2.0): Less dropping, but higher memory/compute
- **Dropping**: Hurts quality, especially if gating is poor

In [ ]:
def simulate_capacity_dropping(num_tokens: int, num_experts: int, capacity_factor: float, logits: np.ndarray):
    """Simulate token dropping with expert capacity constraints."""
    capacity = int(np.ceil((num_tokens / num_experts) * capacity_factor))
    
    # Get top-1 expert for each token
    expert_assignments = np.argmax(logits, axis=1)
    
    # Track expert loads and dropped tokens
    expert_loads = np.zeros(num_experts)
    dropped_mask = np.zeros(num_tokens, dtype=bool)
    
    # Process tokens in order of gating confidence
    confidences = np.max(logits, axis=1)
    sorted_indices = np.argsort(-confidences)  # Highest confidence first
    
    for token_idx in sorted_indices:
        expert_id = expert_assignments[token_idx]
        if expert_loads[expert_id] < capacity:
            expert_loads[expert_id] += 1
        else:
            dropped_mask[token_idx] = True
    
    num_dropped = dropped_mask.sum()
    drop_rate = num_dropped / num_tokens
    
    return expert_loads, dropped_mask, drop_rate, capacity


# Simulate for different capacity factors
num_tokens, num_experts = 1000, 8
np.random.seed(42)
logits = np.random.randn(num_tokens, num_experts)
# Make it imbalanced (some experts more popular)
logits[:, 0] += 0.5  # Expert 0 is more attractive
logits[:, 7] -= 0.5  # Expert 7 is less attractive

capacity_factors = [1.0, 1.25, 1.5, 2.0]
results = []
for cf in capacity_factors:
    loads, dropped, drop_rate, capacity = simulate_capacity_dropping(num_tokens, num_experts, cf, logits)
    results.append((cf, loads, drop_rate, capacity))
    print(f"Capacity factor {cf:.2f}: capacity={capacity}, dropped={int(drop_rate*num_tokens)} ({drop_rate*100:.1f}%)")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Expert loads for different capacity factors
ax = axes[0]
x = np.arange(num_experts)
width = 0.2
for i, (cf, loads, _, _) in enumerate(results):
    ax.bar(x + i*width, loads, width, label=f'CF={cf}', alpha=0.8)
ax.set_xlabel('Expert ID', fontsize=11)
ax.set_ylabel('Number of Tokens', fontsize=11)
ax.set_title('Expert Loads vs Capacity Factor', fontsize=12, fontweight='bold')
ax.set_xticks(x + 1.5*width)
ax.set_xticklabels(x)
ax.legend()
ax.grid(axis='y', alpha=0.3)
# Add capacity line
ideal_load = num_tokens / num_experts
ax.axhline(ideal_load, color='red', linestyle='--', linewidth=1, label=f'Ideal load={ideal_load:.0f}')

# Drop rate vs capacity factor
ax = axes[1]
drop_rates = [r[2] * 100 for r in results]
capacities = [r[3] for r in results]
ax.plot(capacity_factors, drop_rates, marker='o', linewidth=2, markersize=8, color='red')
ax.set_xlabel('Capacity Factor', fontsize=11)
ax.set_ylabel('Drop Rate (%)', fontsize=11)
ax.set_title('Token Drop Rate vs Capacity Factor', fontsize=12, fontweight='bold')
ax.grid(alpha=0.3)
ax.set_xticks(capacity_factors)
# Annotate capacities
for cf, dr, cap in zip(capacity_factors, drop_rates, capacities):
    ax.annotate(f'cap={cap}', (cf, dr), textcoords='offset points', xytext=(0, 10), ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('capacity_dropping.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nHigher capacity factors reduce dropping but increase memory/compute.")

## 3. Switch Transformers

### 3.1 Architecture

**Switch Transformers** (Google, 2021) simplify MoE to k=1:

```
Standard Transformer Layer:
    x = x + Attention(x)
    x = x + FFN(x)

Switch Transformer Layer:
    x = x + Attention(x)
    x = x + MoE_FFN(x)  ← Replace FFN with MoE

MoE_FFN(x):
    expert_id = argmax(W_g · x)
    return Expert[expert_id](x)
```

**Key design choices**:
- **k=1**: Each token routed to exactly 1 expert (simplest, most efficient)
- **Capacity factor**: 1.0-1.25 (tight capacity to force load balancing)
- **Expert placement**: Only in FFN layers (attention remains dense)
- **Scale**: 1.6T parameters (128 experts × 13B per expert)

**Benefits over top-k MoE**:
- Simpler routing (no weighted sum, just expert_id)
- Lower compute per token (1 expert vs 2)
- Lower communication cost (no concatenation)
- Easier to implement and debug

**Challenges**:
- Load balancing critical (k=1 means no fallback)
- Token dropping hurts more (no second expert)
- Routing quality matters more

In [ ]:
class SwitchFFN(nn.Module):
    """Switch Transformer FFN layer with MoE (k=1)."""
    def __init__(self, d_model: int, d_ff: int, num_experts: int, capacity_factor: float = 1.25):
        super().__init__()
        self.d_model = d_model
        self.num_experts = num_experts
        self.capacity_factor = capacity_factor
        
        # Experts
        self.experts = nn.ModuleList([Expert(d_model, d_ff) for _ in range(num_experts)])
        
        # Gating
        self.w_gate = nn.Linear(d_model, num_experts, bias=False)
    
    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, dict]:
        """
        Args:
            x: Input (batch, seq_len, d_model)
        
        Returns:
            output: Processed output
            aux_info: Dict with load and drop statistics
        """
        batch_size, seq_len, d_model = x.shape
        num_tokens = batch_size * seq_len
        
        # Compute expert capacity
        capacity = int(np.ceil((num_tokens / self.num_experts) * self.capacity_factor))
        
        # Flatten
        x_flat = x.view(-1, d_model)
        
        # Router logits
        router_logits = self.w_gate(x_flat)  # (num_tokens, num_experts)
        router_probs = F.softmax(router_logits, dim=-1)
        
        # Select top-1 expert
        expert_gate, expert_index = torch.max(router_probs, dim=-1)
        
        # Initialize output and tracking
        output = torch.zeros_like(x_flat)
        expert_loads = torch.zeros(self.num_experts, device=x.device)
        tokens_dropped = 0
        
        # Process each expert
        for expert_id in range(self.num_experts):
            # Get tokens assigned to this expert
            expert_mask = (expert_index == expert_id)
            expert_tokens_idx = expert_mask.nonzero(as_tuple=True)[0]
            
            # Apply capacity constraint
            num_expert_tokens = len(expert_tokens_idx)
            if num_expert_tokens > capacity:
                # Keep only top capacity tokens by gate confidence
                expert_gates_for_expert = expert_gate[expert_tokens_idx]
                _, top_capacity_indices = torch.topk(expert_gates_for_expert, k=capacity)
                expert_tokens_idx = expert_tokens_idx[top_capacity_indices]
                tokens_dropped += (num_expert_tokens - capacity)
            
            expert_loads[expert_id] = len(expert_tokens_idx)
            
            if len(expert_tokens_idx) > 0:
                # Get input for this expert
                expert_input = x_flat[expert_tokens_idx]
                
                # Compute expert output
                expert_output = self.experts[expert_id](expert_input)
                
                # Weight by gate and write to output
                expert_gate_values = expert_gate[expert_tokens_idx].unsqueeze(-1)
                output[expert_tokens_idx] = expert_gate_values * expert_output
        
        # Reshape
        output = output.view(batch_size, seq_len, d_model)
        
        # Auxiliary info
        aux_info = {
            'expert_loads': expert_loads,
            'tokens_dropped': tokens_dropped,
            'drop_rate': tokens_dropped / num_tokens,
            'capacity': capacity
        }
        
        return output, aux_info


class SwitchTransformerLayer(nn.Module):
    """Single Switch Transformer layer."""
    def __init__(self, d_model: int, d_ff: int, num_heads: int, num_experts: int):
        super().__init__()
        self.attention = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.switch_ffn = SwitchFFN(d_model, d_ff, num_experts)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
    
    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, dict]:
        # Attention
        attn_out, _ = self.attention(x, x, x)
        x = self.norm1(x + attn_out)
        
        # Switch FFN
        ffn_out, aux_info = self.switch_ffn(x)
        x = self.norm2(x + ffn_out)
        
        return x, aux_info


# Test Switch Transformer
print("=== Switch Transformer Test ===")
d_model, d_ff, num_heads, num_experts = 512, 2048, 8, 16
batch_size, seq_len = 8, 64

layer = SwitchTransformerLayer(d_model, d_ff, num_heads, num_experts).to(device)
x = torch.randn(batch_size, seq_len, d_model).to(device)

output, aux = layer(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"\nExpert statistics:")
print(f"  Capacity: {aux['capacity']}")
print(f"  Tokens dropped: {aux['tokens_dropped']} ({aux['drop_rate']*100:.2f}%)")
print(f"  Load balance: min={aux['expert_loads'].min():.0f}, max={aux['expert_loads'].max():.0f}, std={aux['expert_loads'].std():.2f}")
print(f"\nExpert loads: {aux['expert_loads'].cpu().numpy().astype(int)}")

# Calculate sparsity
total_expert_params = sum(p.numel() for p in layer.switch_ffn.experts.parameters())
active_per_token = total_expert_params / num_experts  # k=1
sparsity = 1 - (active_per_token / total_expert_params)
print(f"\nSparsity: {sparsity*100:.1f}% of expert parameters inactive per token")

## 4. Load Balancing

### 4.1 The Load Balancing Problem

**Problem**: Without constraints, gating network may:
- Route all tokens to a few "favorite" experts
- Leave other experts unused ("expert collapse")
- Result: Wasted parameters, poor performance

**Example**: With 8 experts:
- Ideal: Each expert processes 100/8 = 12.5 tokens
- Bad: Expert 0 gets 80 tokens, others get 0-5 each

**Solution**: Add **load balancing loss** to encourage uniform distribution.

### 4.2 Load Balancing Loss

**Two complementary losses**:

**1. Importance Loss** (encourage even expert selection):
```
importance[i] = Σ P(expert=i | token)  ← Sum of probabilities
target = N / E  ← Ideal load per expert

L_importance = CV(importance)²  ← Coefficient of variation
             = (std(importance) / mean(importance))²
```

Minimizes variance in how often experts are selected.

**2. Load Loss** (encourage even token assignment):
```
load[i] = Number of tokens actually assigned to expert i

L_load = CV(load)²
```

Minimizes variance in actual tokens processed.

**Combined loss**:
```
L_total = L_task + α × (L_importance + L_load)

where α ∈ [0.01, 0.1] is the balance coefficient
```

**In Switch Transformers** (simplified):
```
f_i = fraction of tokens routed to expert i
P_i = mean probability assigned to expert i

L_balance = E × Σ f_i × P_i

This equals N/E when perfectly balanced.
```

In [ ]:
def compute_load_balancing_loss(router_probs: torch.Tensor, expert_indices: torch.Tensor, num_experts: int) -> torch.Tensor:
    """
    Compute load balancing loss for MoE.
    
    Args:
        router_probs: Softmax probabilities for all experts (num_tokens, num_experts)
        expert_indices: Selected expert for each token (num_tokens,)
        num_experts: Total number of experts
    
    Returns:
        loss: Load balancing loss (scalar)
    """
    num_tokens = router_probs.shape[0]
    
    # Importance: mean probability for each expert across all tokens
    importance = router_probs.mean(dim=0)  # (num_experts,)
    
    # Load: fraction of tokens assigned to each expert
    load = torch.bincount(expert_indices, minlength=num_experts).float() / num_tokens  # (num_experts,)
    
    # Switch Transformer load balancing loss
    # Encourages: load[i] ∝ importance[i]
    loss = num_experts * (load * importance).sum()
    
    return loss


def coefficient_of_variation(x: torch.Tensor) -> torch.Tensor:
    """Compute CV = std / mean."""
    return x.std() / (x.mean() + 1e-10)


# Demonstrate load balancing loss
print("=== Load Balancing Loss Demo ===")
num_tokens, num_experts = 1000, 8

# Scenario 1: Balanced routing
router_probs_balanced = torch.softmax(torch.randn(num_tokens, num_experts), dim=-1)
expert_indices_balanced = torch.argmax(router_probs_balanced, dim=-1)
loss_balanced = compute_load_balancing_loss(router_probs_balanced, expert_indices_balanced, num_experts)
load_balanced = torch.bincount(expert_indices_balanced, minlength=num_experts).float()

# Scenario 2: Imbalanced routing (favorite expert 0)
router_logits_imbalanced = torch.randn(num_tokens, num_experts)
router_logits_imbalanced[:, 0] += 2.0  # Bias towards expert 0
router_probs_imbalanced = torch.softmax(router_logits_imbalanced, dim=-1)
expert_indices_imbalanced = torch.argmax(router_probs_imbalanced, dim=-1)
loss_imbalanced = compute_load_balancing_loss(router_probs_imbalanced, expert_indices_imbalanced, num_experts)
load_imbalanced = torch.bincount(expert_indices_imbalanced, minlength=num_experts).float()

# Scenario 3: Collapsed routing (all to expert 0)
router_logits_collapsed = torch.randn(num_tokens, num_experts)
router_logits_collapsed[:, 0] += 10.0  # Strong bias
router_probs_collapsed = torch.softmax(router_logits_collapsed, dim=-1)
expert_indices_collapsed = torch.argmax(router_probs_collapsed, dim=-1)
loss_collapsed = compute_load_balancing_loss(router_probs_collapsed, expert_indices_collapsed, num_experts)
load_collapsed = torch.bincount(expert_indices_collapsed, minlength=num_experts).float()

print(f"\n1. Balanced routing:")
print(f"   Load: {load_balanced.numpy()}")
print(f"   CV: {coefficient_of_variation(load_balanced):.4f}")
print(f"   Loss: {loss_balanced:.4f}")

print(f"\n2. Imbalanced routing:")
print(f"   Load: {load_imbalanced.numpy()}")
print(f"   CV: {coefficient_of_variation(load_imbalanced):.4f}")
print(f"   Loss: {loss_imbalanced:.4f}")

print(f"\n3. Collapsed routing:")
print(f"   Load: {load_collapsed.numpy()}")
print(f"   CV: {coefficient_of_variation(load_collapsed):.4f}")
print(f"   Loss: {loss_collapsed:.4f}")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

scenarios = [
    ('Balanced', load_balanced.numpy(), loss_balanced.item()),
    ('Imbalanced', load_imbalanced.numpy(), loss_imbalanced.item()),
    ('Collapsed', load_collapsed.numpy(), loss_collapsed.item())
]

for ax, (title, load, loss) in zip(axes, scenarios):
    x = np.arange(num_experts)
    bars = ax.bar(x, load, color='steelblue', alpha=0.7)
    ax.axhline(num_tokens / num_experts, color='red', linestyle='--', linewidth=2, label='Ideal')
    ax.set_xlabel('Expert ID', fontsize=11)
    ax.set_ylabel('Number of Tokens', fontsize=11)
    ax.set_title(f'{title}\nLoss={loss:.3f}', fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('load_balancing.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nLoad balancing loss penalizes imbalanced routing, preventing expert collapse.")

### 4.3 Training with Load Balancing

In [ ]:
class MoEWithLoadBalancing(nn.Module):
    """MoE layer with integrated load balancing loss."""
    def __init__(self, d_model: int, d_ff: int, num_experts: int, k: int = 2, balance_coef: float = 0.01):
        super().__init__()
        self.num_experts = num_experts
        self.k = k
        self.balance_coef = balance_coef
        
        self.experts = nn.ModuleList([Expert(d_model, d_ff) for _ in range(num_experts)])
        self.gate = TopKGating(d_model, num_experts, k, noisy_gating=True)
    
    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Returns:
            output: MoE output
            aux_loss: Load balancing loss
        """
        batch_size, seq_len, d_model = x.shape
        x_flat = x.view(-1, d_model)
        
        # Get routing (need full probs for load balancing loss)
        # Get logits before top-k
        router_logits = self.gate.w_gate(x_flat)  # (batch*seq, num_experts)
        router_probs = F.softmax(router_logits, dim=-1)
        
        # Top-k gating
        gates, indices, _ = self.gate(x, train=self.training)
        gates_flat = gates.view(-1, self.k)
        indices_flat = indices.view(-1, self.k)
        
        # Compute output (same as SimpleMoE)
        output = torch.zeros_like(x_flat)
        for i in range(self.k):
            expert_indices = indices_flat[:, i]
            expert_weights = gates_flat[:, i:i+1]
            for expert_id in range(self.num_experts):
                mask = (expert_indices == expert_id)
                if mask.any():
                    expert_input = x_flat[mask]
                    expert_output = self.experts[expert_id](expert_input)
                    output[mask] += expert_weights[mask] * expert_output
        
        output = output.view(batch_size, seq_len, d_model)
        
        # Compute load balancing loss
        top1_expert_indices = torch.argmax(router_probs, dim=-1)
        aux_loss = compute_load_balancing_loss(router_probs, top1_expert_indices, self.num_experts)
        
        return output, self.balance_coef * aux_loss


# Training simulation
print("=== Training MoE with Load Balancing ===")
d_model, d_ff, num_experts, k = 256, 1024, 8, 2
batch_size, seq_len = 16, 32

model = MoEWithLoadBalancing(d_model, d_ff, num_experts, k, balance_coef=0.01).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# Dummy task: predict next token (simplified)
vocab_size = 1000
projection = nn.Linear(d_model, vocab_size).to(device)

num_steps = 100
task_losses = []
aux_losses = []
expert_loads_history = []

for step in range(num_steps):
    # Generate dummy data
    x = torch.randn(batch_size, seq_len, d_model).to(device)
    targets = torch.randint(0, vocab_size, (batch_size, seq_len)).to(device)
    
    # Forward
    moe_output, aux_loss = model(x)
    logits = projection(moe_output)
    
    # Task loss (cross-entropy)
    task_loss = F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1))
    
    # Total loss
    total_loss = task_loss + aux_loss
    
    # Backward
    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()
    
    # Track
    task_losses.append(task_loss.item())
    aux_losses.append(aux_loss.item())
    
    # Track expert loads every 10 steps
    if step % 10 == 0:
        with torch.no_grad():
            _, indices, _ = model.gate(x)
            loads = torch.bincount(indices.flatten(), minlength=num_experts).float().cpu().numpy()
            expert_loads_history.append(loads)
    
    if step % 20 == 0:
        print(f"Step {step:3d}: task_loss={task_loss.item():.4f}, aux_loss={aux_loss.item():.4f}, total={total_loss.item():.4f}")

# Visualize training
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Loss curves
ax = axes[0]
ax.plot(task_losses, label='Task Loss', linewidth=2)
ax.plot(aux_losses, label='Aux Loss (×100)', linewidth=2)
ax.set_xlabel('Training Step', fontsize=11)
ax.set_ylabel('Loss', fontsize=11)
ax.set_title('Training Losses', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# Expert loads over time
ax = axes[1]
expert_loads_history = np.array(expert_loads_history).T  # (num_experts, num_snapshots)
for i in range(num_experts):
    ax.plot(expert_loads_history[i], label=f'Expert {i}', linewidth=1.5, alpha=0.7)
ax.set_xlabel('Training Step (×10)', fontsize=11)
ax.set_ylabel('Number of Tokens', fontsize=11)
ax.set_title('Expert Loads Over Training', fontsize=12, fontweight='bold')
ax.legend(ncol=2, fontsize=8)
ax.grid(alpha=0.3)

# Final expert load distribution
ax = axes[2]
final_loads = expert_loads_history[:, -1]
x = np.arange(num_experts)
bars = ax.bar(x, final_loads, color='steelblue', alpha=0.7)
mean_load = final_loads.mean()
ax.axhline(mean_load, color='red', linestyle='--', linewidth=2, label=f'Mean={mean_load:.1f}')
ax.set_xlabel('Expert ID', fontsize=11)
ax.set_ylabel('Number of Tokens', fontsize=11)
ax.set_title('Final Expert Loads', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('moe_training.png', dpi=150, bbox_inches='tight')
plt.show()

cv_initial = coefficient_of_variation(torch.tensor(expert_loads_history[:, 0]))
cv_final = coefficient_of_variation(torch.tensor(expert_loads_history[:, -1]))
print(f"\nLoad balance improvement: CV {cv_initial:.3f} → {cv_final:.3f}")

## 5. Training Challenges

### 5.1 Common Issues

**1. Expert Collapse**:
- Some experts dominate, others unused
- Solution: Strong load balancing loss, noisy gating for exploration

**2. Routing Instability**:
- Expert assignments change drastically between steps
- Causes training instability (different experts each batch)
- Solution: Careful learning rate scheduling, gradient clipping

**3. Communication Overhead**:
- In distributed training, tokens routed across GPUs
- All-to-all communication expensive
- Solution: Expert parallelism (each GPU has subset of experts)

**4. Memory Imbalance**:
- Different GPUs process different numbers of tokens
- Some GPUs may run out of memory
- Solution: Capacity constraints, token dropping

**5. Gradient Conflicts**:
- Task loss wants expert specialization
- Load balancing loss wants uniformity
- Solution: Tune balance coefficient α carefully

### 5.2 Best Practices

**Architecture**:
- Start with k=2 (more robust than k=1)
- Use capacity_factor=1.25-1.5 (balance efficiency and robustness)
- Add noisy gating for exploration

**Training**:
- Warmup without load balancing (first 1000 steps)
- Gradually increase balance coefficient α
- Use lower learning rate for gating network
- Gradient clipping (especially for gating)

**Monitoring**:
- Track expert utilization (CV of loads)
- Monitor drop rate (should be <5%)
- Check routing entropy (high = diverse routing)
- Visualize expert specialization (what do they learn?)

In [ ]:
# Metrics for monitoring MoE training
def compute_routing_entropy(router_probs: torch.Tensor) -> torch.Tensor:
    """Compute entropy of routing distribution (high = diverse)."""
    # Average entropy per token
    entropy = -(router_probs * torch.log(router_probs + 1e-10)).sum(dim=-1).mean()
    return entropy


def compute_expert_utilization(loads: torch.Tensor) -> dict:
    """Compute various expert utilization metrics."""
    loads_np = loads.cpu().numpy()
    return {
        'cv': float(coefficient_of_variation(loads)),
        'min': float(loads.min()),
        'max': float(loads.max()),
        'mean': float(loads.mean()),
        'std': float(loads.std()),
        'num_unused': int((loads == 0).sum()),
        'gini': float(np.abs(np.subtract.outer(loads_np, loads_np)).mean() / (2 * loads_np.mean() + 1e-10))
    }


# Demonstrate metrics
print("=== MoE Training Metrics ===")
num_tokens, num_experts = 1000, 8

# Good routing (balanced)
router_probs_good = torch.softmax(torch.randn(num_tokens, num_experts), dim=-1)
expert_indices_good = torch.argmax(router_probs_good, dim=-1)
loads_good = torch.bincount(expert_indices_good, minlength=num_experts).float()
entropy_good = compute_routing_entropy(router_probs_good)
util_good = compute_expert_utilization(loads_good)

# Bad routing (collapsed)
router_logits_bad = torch.randn(num_tokens, num_experts)
router_logits_bad[:, 0] += 5.0
router_probs_bad = torch.softmax(router_logits_bad, dim=-1)
expert_indices_bad = torch.argmax(router_probs_bad, dim=-1)
loads_bad = torch.bincount(expert_indices_bad, minlength=num_experts).float()
entropy_bad = compute_routing_entropy(router_probs_bad)
util_bad = compute_expert_utilization(loads_bad)

print("\n1. Good Routing (Balanced):")
print(f"   Entropy: {entropy_good:.3f}")
print(f"   CV: {util_good['cv']:.3f}")
print(f"   Gini: {util_good['gini']:.3f}")
print(f"   Unused: {util_good['num_unused']}/{num_experts}")
print(f"   Load range: [{util_good['min']:.0f}, {util_good['max']:.0f}]")

print("\n2. Bad Routing (Collapsed):")
print(f"   Entropy: {entropy_bad:.3f}")
print(f"   CV: {util_bad['cv']:.3f}")
print(f"   Gini: {util_bad['gini']:.3f}")
print(f"   Unused: {util_bad['num_unused']}/{num_experts}")
print(f"   Load range: [{util_bad['min']:.0f}, {util_bad['max']:.0f}]")

print("\n📊 Target metrics for healthy MoE training:")
print("   • Entropy: >1.5 (higher = more diverse routing)")
print("   • CV: <0.3 (lower = more balanced)")
print("   • Gini: <0.2 (lower = more equal)")
print("   • Unused experts: 0")
print("   • Drop rate: <5%")

## 6. Advanced MoE Variants

### 6.1 Expert Choice Routing

**Idea**: Instead of tokens choosing experts, **experts choose tokens**.

```
Standard (token-choice):
    For each token:
        scores = softmax(W_g · token)
        expert = topK(scores, k)
    
Expert-choice:
    For each expert:
        scores = W_expert · all_tokens  ← Expert computes affinity to all tokens
        selected_tokens = topK(scores, capacity)
        process those tokens
```

**Benefits**:
- Perfect load balancing (each expert processes exactly `capacity` tokens)
- No token dropping (every token processed by some expert)
- Simpler training (no load balancing loss needed)

**Trade-offs**:
- More communication (experts need global view of all tokens)
- Less token-specific routing (tokens can't "choose" their expert)

### 6.2 Mixture-of-Depths (MoD)

**Idea**: Instead of sparse **width** (subset of experts), use sparse **depth** (subset of layers).

```
For each token:
    decide = Router(token)  ← Binary: process or skip this layer
    if decide == process:
        token = Layer(token)
    else:
        token = token  (skip, pass through)
```

**Benefits**:
- Adaptive computation (easy tokens use fewer layers)
- No communication overhead (decision is per-token)
- Can combine with MoE (sparse width + sparse depth)

### 6.3 Soft MoE

**Idea**: Instead of routing tokens to experts, route **weighted combinations** of all tokens to each expert.

```
For each expert i:
    # Compute attention-like weights over all tokens
    weights_i = softmax(score(expert_i, all_tokens))
    
    # Weighted average of inputs
    input_i = Σⱼ weights_i[j] · token[j]
    
    # Process
    output_i = Expert_i(input_i)

# Reconstruct per-token outputs by reversing the weighting
output[j] = Σᵢ weights_i[j] · output_i
```

**Benefits**:
- Differentiable (no discrete routing)
- No load balancing issues (all experts always used)
- Good for small-scale MoE

**Trade-offs**:
- Less sparse (all experts always active)
- More compute than hard routing

### 6.4 MoE in Different Architectures

**Vision Transformers**:
- MoE in FFN layers (like NLP)
- Can also apply to patch embeddings
- Experts can specialize by visual concept (edges, textures, objects)

**Multimodal**:
- Expert per modality (text, image, audio)
- Cross-modal experts for fusion
- Router learns when to use which modality

**Decoder-only LLMs**:
- Replace FFN with MoE (like Switch)
- Keep attention dense (routing on queries/keys is harder)
- Can use different num_experts per layer

In [ ]:
# Simple Expert-Choice implementation
class ExpertChoiceRouting(nn.Module):
    """Expert-choice routing: experts select tokens."""
    def __init__(self, d_model: int, num_experts: int, capacity_per_expert: int):
        super().__init__()
        self.num_experts = num_experts
        self.capacity = capacity_per_expert
        
        # Each expert has a query vector to score tokens
        self.expert_queries = nn.Parameter(torch.randn(num_experts, d_model))
    
    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            x: Tokens (batch, seq_len, d_model)
        
        Returns:
            assignment: Binary (num_experts, num_tokens)
            gates: Weights for selected tokens
        """
        batch_size, seq_len, d_model = x.shape
        num_tokens = batch_size * seq_len
        x_flat = x.view(num_tokens, d_model)
        
        # Compute scores: (num_experts, num_tokens)
        scores = torch.matmul(self.expert_queries, x_flat.T)  # (num_experts, num_tokens)
        
        # Each expert selects top-k tokens
        assignment = torch.zeros_like(scores)
        gates = torch.zeros_like(scores)
        
        for expert_id in range(self.num_experts):
            expert_scores = scores[expert_id]
            top_k_values, top_k_indices = torch.topk(expert_scores, k=self.capacity)
            
            # Softmax over selected tokens
            top_k_gates = F.softmax(top_k_values, dim=0)
            
            assignment[expert_id, top_k_indices] = 1
            gates[expert_id, top_k_indices] = top_k_gates
        
        return assignment, gates


# Demo
print("=== Expert-Choice Routing Demo ===")
d_model, num_experts = 256, 8
batch_size, seq_len = 4, 16
num_tokens = batch_size * seq_len
capacity = num_tokens // num_experts  # Balanced capacity

router = ExpertChoiceRouting(d_model, num_experts, capacity)
x = torch.randn(batch_size, seq_len, d_model)

assignment, gates = router(x)

print(f"Input: {x.shape}")
print(f"Assignment: {assignment.shape}")
print(f"\nTokens per expert: {assignment.sum(dim=1).numpy()}")
print(f"Experts per token: {assignment.sum(dim=0).numpy()}")
print(f"\n✓ Perfect load balancing: Every expert processes exactly {capacity} tokens")
print(f"✓ No dropping: Every token processed by {assignment.sum(dim=0)[0]:.0f} expert(s)")

# Visualize
plt.figure(figsize=(10, 6))
plt.imshow(assignment.detach().numpy(), aspect='auto', cmap='Reds', interpolation='none')
plt.xlabel('Token Index', fontsize=11)
plt.ylabel('Expert ID', fontsize=11)
plt.title('Expert-Choice Routing Assignment', fontsize=12, fontweight='bold')
plt.colorbar(label='Selected')
plt.tight_layout()
plt.savefig('expert_choice.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Exercises

### Exercise 1: Implement capacity-aware Switch routing
Modify `SwitchFFN` to properly handle capacity constraints and track dropped tokens.

### Exercise 2: Analyze expert specialization
Train a MoE on a simple classification task (e.g., MNIST) and visualize what each expert learns. Do they specialize by digit?

### Exercise 3: Compare routing strategies
Implement and compare:
- Top-1 (Switch)
- Top-2 (standard MoE)
- Soft routing (all experts)
- Expert-choice

Measure: accuracy, compute cost, load balance, training stability.

### Exercise 4: Implement noisy top-k gating
Add tunable Gaussian noise to gating logits during training. Study the effect on:
- Expert utilization
- Final performance
- Training stability

### Exercise 5: Design a load balancing curriculum
Start with high balance coefficient α, gradually decrease during training. Compare to:
- Fixed high α
- Fixed low α
- Increasing α

### Exercise 6: MoE for adaptation
Train a base model, then add MoE layers to adapt to a new domain. Compare to:
- Full fine-tuning
- LoRA
- Adding dense layers

### Exercise 7: Analyze communication costs
Simulate distributed MoE training:
- 8 GPUs, 64 experts (8 per GPU)
- Measure: all-to-all communication volume
- How does it scale with batch size, sequence length, num_experts?

### Exercise 8: Implement Soft MoE
Build a Soft MoE layer where experts process weighted combinations of all tokens. Compare to hard routing.

---

## Key Takeaways

1. **MoE enables efficient scaling**: Add parameters without proportional compute
2. **Load balancing is critical**: Auxiliary loss prevents expert collapse
3. **Capacity constraints manage memory**: But may require dropping tokens
4. **k=1 (Switch) vs k=2**: Trade-off between simplicity and robustness
5. **Training challenges**: Routing instability, communication, gradient conflicts
6. **Variants**: Expert-choice, Soft MoE, Mixture-of-Depths for different needs
7. **GPT-4 scale**: Likely uses MoE to reach trillion-parameter scale efficiently

**MoE is the key to training massive models like GPT-4, Mixtral, and Switch Transformers with practical compute budgets!**